# EvoTumor — Size / Sphericity Sweep Test

Small controlled sweep to sanity-check that the mask + tumor diffusion
models respond sensibly to:
  1. increasing tumor SIZE (volume + consistent linear-extent features)
  2. increasing tumor SPHERICITY (shape roundness, holding volume fixed)

Assumes this notebook runs from the same working directory / PYTHONPATH
as your hydra entrypoint (i.e. `TumorGeneration` importable, `config/`
discoverable by hydra).

In [1]:
import os
os.chdir("..")

In [2]:
import copy
import math
import os

import numpy as np
import torch
import nibabel as nib
from omegaconf import DictConfig
import hydra
from hydra import compose, initialize_config_dir

from TumorGeneration.tumor_gen_utils import (
    MASK_COLUMNS,
    TUMOR_COLUMNS,
    ORGAN_TO_IDX,
    sample_radiomics,
    prepare_mask_model,
    prepare_tumor_model,
    synthesize_tumor,
    get_size,
)
from TumorGeneration.radiomics_sampler.utils import apply_normalization as apply_radiomics_normalization
from dataset.dataloader import get_healthy_loader

## 1. Load config via Hydra (same config as your training/inference script)

`synthesis.yaml` is the same config_name used in `generate_samples.py`.
Using `initialize_config_dir` (absolute path) rather than the
`@hydra.main` decorator since we're in a notebook, not a CLI entrypoint.

In [3]:
CONFIG_DIR = os.path.abspath("config")  # adjust if your config/ lives elsewhere

with initialize_config_dir(config_dir=CONFIG_DIR, version_base=None):
    cfg: DictConfig = compose(config_name="synthesis")

print(cfg)

{'paths': {'tumor_diffusion_ckpt_dir': '/projects/bodymaps/Rohin/TumorSynthesis/STEP3.ImageDiffusionModel/checkpoints/ddpm/multi_tumor_train/tumor_diffusion_all_radiomics_500tsteps_UnetCA_128dim', 'mask_diffusion_ckpt_dir': '/projects/bodymaps/Rohin/TumorSynthesis/STEP2ACTUAL.MaskDiffusionModel/checkpoints/multi_tumor_train/mask_diffusion_all_radiomics_500tsteps_32size_4space_64dim_actual', 'radiomics_gmm_bank': '/projects/bodymaps/Rohin/TumorSynthesis/STEP4.SegmentationModel/TumorGeneration/radiomics_sampler/gmm_bank.pkl'}, 'dataset': {'tumor_norm_stats': '/projects/bodymaps/Rohin/TumorSynthesis/STEP3.ImageDiffusionModel/dataset_norm_stats_tumor_diffusion_all_radiomics_500tsteps_UnetCA_128dim.json', 'mask_norm_stats': '/projects/bodymaps/Rohin/TumorSynthesis/STEP2ACTUAL.MaskDiffusionModel/dataset_norm_stats.json', 'dataset_list': 'abdomen_atlas_pro', 'data_root_path': '/projects/bodymaps/Data/image_only/AbdomenAtlasPro/AbdomenAtlasPro/', 'organ_segmentations_root_path': '/projects/bod

In [4]:
GPU_IDX = cfg.inference.gpu_idxs
device = torch.device(f"cuda:{GPU_IDX}")
torch.cuda.set_device(GPU_IDX)

mask_tester = prepare_mask_model(device, cfg)
tumor_tester = prepare_tumor_model(device, cfg)

gmm_bank_path = cfg.paths.radiomics_gmm_bank

import json
with open(cfg.dataset.tumor_norm_stats) as f:
    tumor_norm_stats = json.load(f)
with open(cfg.dataset.mask_norm_stats) as f:
    mask_norm_stats = json.load(f)

LOADED: Mask Diffusion Model - /projects/bodymaps/Rohin/TumorSynthesis/STEP2ACTUAL.MaskDiffusionModel/checkpoints/multi_tumor_train/mask_diffusion_all_radiomics_500tsteps_32size_4space_64dim_actual/model_best.pt
LOADED: Tumor Diffusion Model - /projects/bodymaps/Rohin/TumorSynthesis/STEP3.ImageDiffusionModel/checkpoints/ddpm/multi_tumor_train/tumor_diffusion_all_radiomics_500tsteps_UnetCA_128dim/model_best.pt


## 2. Grab one real healthy sample to synthesize onto

We just pull a single batch from the existing healthy dataloader so the
CT / organ_mask / heatmap inputs are realistic and match what the models
saw at training time.

In [5]:
healthy_loader, _, _ = get_healthy_loader(cfg.dataset)
batch = next(iter(healthy_loader))

ct = batch["image"].to(device)
organ_mask = batch["organ_mask"].to(device)
m_organ_mask = batch["m_organ_mask"].to(device)
heatmap = batch["heatmap"].to(device)
organ = batch["organ"][0]
bdmap_id = batch["bdmap_id"][0]

print(f"Using bdmap_id={bdmap_id}, organ={organ}")

monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.


train len 160503
SETTING UP PERSISTENT CACHE
[CACHE-BUILD 23:24:01.816] Running 'LoadImageh5d' for pair (? -> ?) — this should print exactly once per pair per persistent cache.[CACHE-BUILD 23:24:01.816] Running 'LoadImageh5d' for pair (? -> ?) — this should print exactly once per pair per persistent cache.[CACHE-BUILD 23:24:01.816] Running 'LoadImageh5d' for pair (? -> ?) — this should print exactly once per pair per persistent cache.[CACHE-BUILD 23:24:01.817] Running 'LoadImageh5d' for pair (? -> ?) — this should print exactly once per pair per persistent cache.[CACHE-BUILD 23:24:01.817] Running 'LoadImageh5d' for pair (? -> ?) — this should print exactly once per pair per persistent cache.
[CACHE-BUILD 23:24:01.817] Running 'LoadImageh5d' for pair (? -> ?) — this should print exactly once per pair per persistent cache.
[CACHE-BUILD 23:24:01.817] Running 'LoadImageh5d' for pair (? -> ?) — this should print exactly once per pair per persistent cache.



[CACHE-BUILD 23:24:01.818] Runni

no available indices of class 1 to crop, setting the crop ratio of this class to zero.


[CACHE-BUILD 23:24:06.922] Running 'LoadImageh5d' for pair (? -> ?) — this should print exactly once per pair per persistent cache.
Using bdmap_id=BDMAP_00381127, organ=bladder


[CACHE-BUILD 23:24:08.513] Running 'LoadImageh5d' for pair (? -> ?) — this should print exactly once per pair per persistent cache.
[CACHE-BUILD 23:24:08.767] Running 'LoadImageh5d' for pair (? -> ?) — this should print exactly once per pair per persistent cache.
[CACHE-BUILD 23:24:09.941] Running 'LoadImageh5d' for pair (? -> ?) — this should print exactly once per pair per persistent cache.
[CACHE-BUILD 23:24:10.066] Running 'LoadImageh5d' for pair (? -> ?) — this should print exactly once per pair per persistent cache.
[CACHE-BUILD 23:24:11.026] Running 'LoadImageh5d' for pair (? -> ?) — this should print exactly once per pair per persistent cache.
[CACHE-BUILD 23:24:11.741] Running 'LoadImageh5d' for pair (? -> ?) — this should print exactly once per pair per persistent cache.
[CACHE-BUILD 23:24:12.135] Running 'LoadImageh5d' for pair (? -> ?) — this should print exactly once per pair per persistent cache.
[CACHE-BUILD 23:24:12.142] Running 'LoadImageh5d' for pair (? -> ?) — this s

no available indices of class 1 to crop, setting the crop ratio of this class to zero.


[CACHE-BUILD 23:24:18.958] Running 'LoadImageh5d' for pair (? -> ?) — this should print exactly once per pair per persistent cache.
[CACHE-BUILD 23:24:20.423] Running 'LoadImageh5d' for pair (? -> ?) — this should print exactly once per pair per persistent cache.
[CACHE-BUILD 23:24:22.012] Running 'LoadImageh5d' for pair (? -> ?) — this should print exactly once per pair per persistent cache.


no available indices of class 1 to crop, setting the crop ratio of this class to zero.
no available indices of class 1 to crop, setting the crop ratio of this class to zero.


## 3. Base radiomics draw (held fixed except for the swept axis)

We draw one real sample from the GMM bank for this organ, so every
non-swept feature (texture/appearance columns, and shape columns we're
not deliberately perturbing) stays realistic and internally consistent.
We then clone + override just the features relevant to the current sweep
step.

In [6]:
base_radiomics = sample_radiomics(gmm_bank_path, organ)
base_mask = base_radiomics["mask_radiomics"]

print("Base mask radiomics (unperturbed draw):")
for k in ["diameter_x_mm", "diameter_y_mm", "diameter_z_mm",
          "original_shape_MeshVolume", "original_shape_VoxelVolume",
          "original_shape_SurfaceArea", "original_shape_Sphericity"]:
    print(f"  {k}: {base_mask[k]:.3f}")

Base mask radiomics (unperturbed draw):
  diameter_x_mm: 6.826
  diameter_y_mm: 5.740
  diameter_z_mm: 1.112
  original_shape_MeshVolume: 25.046
  original_shape_VoxelVolume: 28.146
  original_shape_SurfaceArea: 74.527
  original_shape_Sphericity: 0.641


X does not have valid feature names, but PowerTransformer was fitted with feature names


## 4. Size sweep helper

`get_size()` buckets on `original_shape_VoxelVolume` alone, but a
consistent tumor of a given volume also has consistent diameters / axis
lengths / surface area. Scaling volume alone while leaving those small
produces an internally-inconsistent radiomics vector outside the training
distribution. So here we treat the tumor as scaling like a similar
ellipsoid by a linear factor `s`:

  volume        ~ s^3
  linear extents (diameters, axis lengths) ~ s
  surface area  ~ s^2

and rescale every relevant MASK_COLUMNS field consistently from the base
draw using a single linear scale factor `s`.

In [7]:
_LINEAR_FIELDS = [
    "diameter_x_mm", "diameter_y_mm", "diameter_z_mm",
    "original_shape_LeastAxisLength", "original_shape_MajorAxisLength",
    "original_shape_MinorAxisLength",
    "original_shape_Maximum2DDiameterColumn", "original_shape_Maximum2DDiameterRow",
    "original_shape_Maximum2DDiameterSlice", "original_shape_Maximum3DDiameter",
]
_AREA_FIELDS = ["original_shape_SurfaceArea"]
_VOLUME_FIELDS = ["original_shape_MeshVolume", "original_shape_VoxelVolume"]
# Elongation / Flatness / Sphericity / SurfaceVolumeRatio are shape-normalized
# ratios and are left untouched by a pure size scale (a bigger tumor of the
# same shape has the same elongation/flatness/sphericity).
# SurfaceVolumeRatio does change under isotropic scaling (~ 1/s), handled below.


def make_size_variant(base_mask_dict, scale_factor):
    """Scale a mask-radiomics dict as if the tumor grew by linear factor
    `scale_factor`, keeping shape ratios (elongation, flatness, sphericity)
    fixed."""
    m = copy.deepcopy(base_mask_dict)
    for k in _LINEAR_FIELDS:
        m[k] = base_mask_dict[k] * scale_factor
    for k in _AREA_FIELDS:
        m[k] = base_mask_dict[k] * (scale_factor ** 2)
    for k in _VOLUME_FIELDS:
        m[k] = base_mask_dict[k] * (scale_factor ** 3)
    # surface_area / volume ratio scales as 1/s under isotropic scaling
    m["original_shape_SurfaceVolumeRatio"] = (
        base_mask_dict["original_shape_SurfaceVolumeRatio"] / scale_factor
    )
    return m

## 5. Sphericity sweep helper

Sphericity is bounded in (0, 1] and is literally derived from volume and
surface area (sphericity = (36*pi*V^2)^(1/3) / SurfaceArea, up to the
exact pyradiomics convention). To sweep sphericity independently, we hold
`VoxelVolume`/`MeshVolume` FIXED and back out the surface area implied by
the target sphericity, so the (volume, surface_area, sphericity) triple
stays internally consistent instead of just overwriting the Sphericity
field in isolation (which is exactly the kind of inconsistency you flagged
as producing the negative-R^2 artifact).

In [8]:
def make_sphericity_variant(base_mask_dict, target_sphericity):
    """Hold volume fixed, solve for the surface area implied by
    target_sphericity, and overwrite Sphericity + SurfaceArea +
    SurfaceVolumeRatio consistently. Elongation/Flatness (independent shape
    descriptors) are left as-is from the base draw."""
    m = copy.deepcopy(base_mask_dict)
    volume = base_mask_dict["original_shape_MeshVolume"]

    # sphericity = (36 * pi * V^2)^(1/3) / SA  =>  SA = (36*pi*V^2)^(1/3) / sphericity
    ideal_numerator = (36.0 * math.pi * (volume ** 2)) ** (1.0 / 3.0)
    target_sphericity = min(max(target_sphericity, 1e-3), 1.0)  # keep in (0, 1]
    implied_surface_area = ideal_numerator / target_sphericity

    m["original_shape_Sphericity"] = target_sphericity
    m["original_shape_SurfaceArea"] = implied_surface_area
    m["original_shape_SurfaceVolumeRatio"] = implied_surface_area / volume
    return m

## 6. Combine size + sphericity into a single joint variant

For the grid, each cell is a (volume_scale, sphericity_target) pair.
`make_joint_variant` applies the size scaling first (which moves volume,
diameters, surface area together) and then re-solves surface area for the
target sphericity *at that scaled volume*, so every cell has an
internally-consistent (volume, surface_area, sphericity) triple.

In [9]:
def make_joint_variant(base_mask_dict, scale_factor, target_sphericity):
    sized = make_size_variant(base_mask_dict, scale_factor)
    joint = make_sphericity_variant(sized, target_sphericity)
    return joint

## 7. Define the sweep grid and run it

Set `VOLUME_SCALES` and `SPHERICITY_TARGETS` to whatever lists you want —
the grid is `len(VOLUME_SCALES) x len(SPHERICITY_TARGETS)` cells, each one
a full mask + tumor-appearance synthesis. Everything outside these two
axes (organ, base CT, texture/appearance radiomics, non-swept shape
features) is held fixed to the single base draw from Section 3, so the
grid isolates just these two effects.

In [13]:
VOLUME_SCALES = [0.6, 1.0, 1.4, 1.8]       # linear scale factors (not raw volume multipliers)
SPHERICITY_TARGETS = [0.55, 0.70, 0.85, 0.95]  # low (irregular) -> near-spherical

OUT_DIR = "sweep_outputs"
os.makedirs(OUT_DIR, exist_ok=True)


def run_one(mask_radiomics_override, tag):
    """Runs mask + tumor synthesis for one radiomics override, mirroring
    synthesize_tumor's body but skipping its internal GMM draw (step 1) so we
    can inject our own controlled mask_radiomics."""
    from TumorGeneration.tumor_gen_utils import (
        generate_tumor_mask, sample_tumor_appearance,
    )
    from scipy.ndimage import gaussian_filter

    radiomics = {
        "mask_radiomics": mask_radiomics_override,
        "tumor_radiomics": copy.deepcopy(base_radiomics["tumor_radiomics"]),
    }
    normalized_radiomics = apply_radiomics_normalization(
        radiomics, tumor_norm_stats, mask_norm_stats
    )
    print(normalized_radiomics)
    tumor_mask = generate_tumor_mask(
        mask_tester, organ, m_organ_mask, organ_mask, heatmap,
        normalized_radiomics, device,
    )
    print(f"[{tag}] tumor mask voxels: {tumor_mask.sum().item():.0f}  "
          f"(bucket={get_size(radiomics)})")

    return final_hu, tumor_mask, radiomics


def save_result(tag, final_volume, tumor_mask):
    affine = np.eye(4)
    vol_np = final_volume[0, 0].detach().cpu().numpy().astype(np.float32)
    mask_np = tumor_mask[0, 0].detach().cpu().numpy().astype(np.uint8)
    nib.save(nib.Nifti1Image(vol_np, affine), os.path.join(OUT_DIR, f"{tag}_ct.nii.gz"))
    nib.save(nib.Nifti1Image(mask_np, affine), os.path.join(OUT_DIR, f"{tag}_mask.nii.gz"))
    return vol_np, mask_np

In [15]:
# grid[row][col] -> dict with numpy CT/mask slices + metadata, row = volume
# scale index, col = sphericity target index
grid = [[None for _ in SPHERICITY_TARGETS] for _ in VOLUME_SCALES]

for i, scale in enumerate(VOLUME_SCALES):
    for j, sph in enumerate(SPHERICITY_TARGETS):
        variant = make_joint_variant(base_mask, scale, sph)
        tag = f"vol{scale:.2f}_sph{sph:.2f}"
        final_vol, tmask, rad = run_one(variant, tag)
        raise RuntimeError("")
        vol_np, mask_np = save_result(tag, final_vol, tmask)
        grid[i][j] = {
            "tag": tag,
            "scale": scale,
            "target_sphericity": sph,
            "voxel_volume": variant["original_shape_VoxelVolume"],
            "achieved_mask_voxels": int(tmask.sum().item()),
            "ct": vol_np,
            "mask": mask_np,
        }

print("Grid synthesis complete:", len(VOLUME_SCALES), "x", len(SPHERICITY_TARGETS))

{'mask_radiomics': {'diameter_x_mm': -1.0136932131436256, 'diameter_y_mm': -1.036992409930193, 'diameter_z_mm': -0.9943747007586567, 'original_shape_Elongation': 0.4525658864729288, 'original_shape_Flatness': -1.293438431414049, 'original_shape_LeastAxisLength': -1.0275545082544242, 'original_shape_MajorAxisLength': -1.0559012730564943, 'original_shape_Maximum2DDiameterColumn': -1.07497057525286, 'original_shape_Maximum2DDiameterRow': -1.0762732498501608, 'original_shape_Maximum2DDiameterSlice': -1.0341255411770436, 'original_shape_Maximum3DDiameter': -1.0966922914651556, 'original_shape_MeshVolume': -0.24228696296702426, 'original_shape_MinorAxisLength': -1.0342962089634713, 'original_shape_Sphericity': -0.5842242264726267, 'original_shape_SurfaceArea': -0.4512579425481922, 'original_shape_SurfaceVolumeRatio': 6.129382358409768, 'original_shape_VoxelVolume': -0.24242582474086113}, 'tumor_radiomics': {'attenuation_delta': -0.09290507412839323, 'original_firstorder_10Percentile': 0.3170

torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
None of the inputs have requires_grad=True. Gradients will be None


KeyboardInterrupt: 

## 8. Figure: grid of volume x sphericity

Rows = volume scale (increasing downward), columns = sphericity target
(increasing rightward). Each cell shows the axial slice through the
tumor's center of mass, CT windowed to soft-tissue range, with the
synthesized tumor mask outlined in red.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch


def center_of_mass_slice(mask_np):
    """Axial (Z) index through the mask's center of mass; falls back to the
    volume's mid-slice if the mask is empty."""
    if mask_np.sum() == 0:
        return mask_np.shape[-1] // 2
    zs = np.nonzero(mask_np)[-1]
    return int(np.round(zs.mean()))


def window_ct(slice_2d, wl=40, ww=400):
    """Simple soft-tissue CT windowing for display (window level/width)."""
    lo, hi = wl - ww / 2, wl + ww / 2
    return np.clip(slice_2d, lo, hi)


n_rows, n_cols = len(VOLUME_SCALES), len(SPHERICITY_TARGETS)
fig, axes = plt.subplots(
    n_rows, n_cols, figsize=(3.2 * n_cols, 3.2 * n_rows), squeeze=False
)

for i in range(n_rows):
    for j in range(n_cols):
        cell = grid[i][j]
        ax = axes[i][j]

        z = center_of_mass_slice(cell["mask"])
        ct_slice = window_ct(cell["ct"][:, :, z])
        mask_slice = cell["mask"][:, :, z]

        ax.imshow(ct_slice.T, cmap="gray", origin="lower")
        # red mask outline: mask boundary via simple erosion diff
        from scipy.ndimage import binary_erosion
        boundary = mask_slice.astype(bool) & ~binary_erosion(mask_slice.astype(bool))
        overlay = np.zeros((*boundary.T.shape, 4))
        overlay[boundary.T] = [1, 0, 0, 1]
        ax.imshow(overlay, origin="lower")

        ax.set_xticks([])
        ax.set_yticks([])

        if i == 0:
            ax.set_title(f"sphericity={cell['target_sphericity']:.2f}", fontsize=11)
        if j == 0:
            ax.set_ylabel(f"vol scale={cell['scale']:.2f}", fontsize=11)

        ax.text(
            0.02, 0.02, f"{cell['achieved_mask_voxels']} vox",
            transform=ax.transAxes, color="yellow", fontsize=8, va="bottom",
        )

fig.suptitle(f"EvoTumor sweep — organ={organ}, bdmap_id={bdmap_id}", fontsize=13)
fig.legend(
    handles=[Patch(edgecolor="red", facecolor="none", label="synthesized tumor mask")],
    loc="lower center", ncol=1, bbox_to_anchor=(0.5, -0.02),
)
fig.tight_layout(rect=[0, 0.02, 1, 0.97])

fig_path = os.path.join(OUT_DIR, "sweep_grid.png")
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()

print("Figure saved to:", fig_path)

## 9. Quick sanity checks

- Across a row (fixed volume, increasing sphericity): `achieved_mask_voxels`
  should stay roughly constant, and the mask outline should look
  progressively rounder/more compact left -> right.
- Down a column (fixed sphericity, increasing volume): the mask outline
  should grow, `achieved_mask_voxels` should increase roughly
  monotonically.
- If sphericity doesn't visibly change the outline shape across a row,
  that's consistent with what you've already been seeing (near-zero /
  negative R^2 on Sphericity conditioning) — this figure just makes it a
  direct visual check instead of an aggregate metric.

In [ ]:
print("Outputs written to:", OUT_DIR)
print(os.listdir(OUT_DIR))